This notebook fits orthos to all replicates of the shendure-calibrated simulations

Imports

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

2025-10-10 15:33:32.874772: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-10 15:33:32.879085: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

Autoreload for dev

In [2]:
%load_ext autoreload
%autoreload 2

Create a nice large cluster. We will need the resorces.

In [3]:
cluster=SLURMCluster(
    cores=4,#cores per slurm job
    memory="64G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=4:00:00",
        f"--output=slave_%j.out"]
)
cluster.scale(jobs=6)
client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )

Load the simulation object

In [4]:
DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"
simu_obj=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251008")

In [5]:
#temporary : obj created w/ older version of code, necessitating this. 
simu_obj.orthos=[]

Fit orthos to all replicates

In [6]:
simu_obj.create_orthos_for_all_replicates(client)

scMPRAforge: INFO: Dropped 635027 of 4681279 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 635514 of 4684178 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 637026 of 4691445 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 636156 of 4693425 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 636332 of 4694417 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


Save

In [7]:
simu_obj.save(path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_with_orthos_20251008")

Shut down the cluster

In [16]:
client.close()
cluster.close()